In [1]:
from peft import TaskType

TaskType.TOKEN_CLS?

Type:        TaskType
String form: TaskType.TOKEN_CLS
Length:      9
File:        c:\users\hhm18\miniconda3\envs\trainingcamp\lib\site-packages\peft\utils\peft_types.py
Docstring:  
Enum class for the different types of tasks supported by PEFT.

Overview of the supported task types:
- SEQ_CLS: Text classification.
- SEQ_2_SEQ_LM: Sequence-to-sequence language modeling.
- CAUSAL_LM: Causal language modeling.
- TOKEN_CLS: Token classification.
- QUESTION_ANS: Question answering.
- FEATURE_EXTRACTION: Feature extraction. Provides the hidden states which can be used as embeddings or features
  for downstream tasks.

In [ ]:
from seqeval.metrics import classification_report
from seqeval.scheme import IOB2

y_true = [['O', 'O', 'O', 'B-LOC', 'I-LOC', 'I-LOC', 'B-LOC', 'O'], ['B-PER', 'I-PER', 'O']]
y_pred = [['O', 'O', 'B-LOC', 'I-LOC', 'I-LOC', 'I-LOC', 'B-LOC', 'O'], ['B-PER', 'I-PER', 'O']]

print(classification_report(y_true, y_pred,  mode='strict', scheme=IOB2))

              precision    recall  f1-score   support

         LOC       0.50      0.50      0.50         2
         PER       1.00      1.00      1.00         1

   micro avg       0.67      0.67      0.67         3
   macro avg       0.75      0.75      0.75         3
weighted avg       0.67      0.67      0.67         3



# 数据处理

In [3]:
categories = set()

def load_data(data_file):
    Data = {}
    with open (data_file, "rt", encoding="utf-8") as f:
        # 文本使用空行进行分割句子
        for idx, line in enumerate(f.read().split("\n\n")):
            if not line:
                break
            sentence, labels = "", []
            for i, item in enumerate(line.split("\n")):
                char, tag = item.split(" ")
                sentence += char
                if tag.startswith("B"):
                    labels.append([i, i, char, tag[2:]])   # Remove the B- or I-
                    categories.add(tag[2:])
                elif tag.startswith("I"):
                    labels[-1][1] = i
                    labels[-1][2] += char
            Data[idx] = {
                "sentence" : sentence,
                "labels" : labels
            }
    return Data

In [14]:
path = "./state3/dataset/ner_data/medical.train"

ds = load_data(path)
len(ds), ds[1]

(5259,
 {'sentence': '目的观察复方丁香开胃贴外敷神阙穴治疗慢性心功能不全伴功能性消化不良的临床疗效',
  'labels': [[4, 10, '复方丁香开胃贴', '中医治疗'], [20, 32, '心功能不全伴功能性消化不良', '西医诊断']]})

In [27]:
def span_to_bio(sentence, spans):
    tokens = list(sentence)
    labels = ["O"] * len(tokens)
    for start, end, x, token_type in spans:
        labels[start] = f"B-{token_type}"
        print(f"x: {x}, start & end: {start},{end}, token_type: {token_type}")
        for i in range(start + 1, end + 1):
            labels[i] = f"I-{token_type}"
    return {"tokens": tokens, "labels": labels}

In [28]:
span_to_bio(ds[1]["sentence"], ds[1]["labels"])

x: 复方丁香开胃贴, start & end: 4,10, token_type: 中医治疗
x: 心功能不全伴功能性消化不良, start & end: 20,32, token_type: 西医诊断


{'tokens': ['目',
  '的',
  '观',
  '察',
  '复',
  '方',
  '丁',
  '香',
  '开',
  '胃',
  '贴',
  '外',
  '敷',
  '神',
  '阙',
  '穴',
  '治',
  '疗',
  '慢',
  '性',
  '心',
  '功',
  '能',
  '不',
  '全',
  '伴',
  '功',
  '能',
  '性',
  '消',
  '化',
  '不',
  '良',
  '的',
  '临',
  '床',
  '疗',
  '效'],
 'labels': ['O',
  'O',
  'O',
  'O',
  'B-中医治疗',
  'I-中医治疗',
  'I-中医治疗',
  'I-中医治疗',
  'I-中医治疗',
  'I-中医治疗',
  'I-中医治疗',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'B-西医诊断',
  'I-西医诊断',
  'I-西医诊断',
  'I-西医诊断',
  'I-西医诊断',
  'I-西医诊断',
  'I-西医诊断',
  'I-西医诊断',
  'I-西医诊断',
  'I-西医诊断',
  'I-西医诊断',
  'I-西医诊断',
  'I-西医诊断',
  'O',
  'O',
  'O',
  'O',
  'O']}

In [29]:
from torch.utils.data import Dataset

categories = set()

class MyDataProcFunc(Dataset):
    def __init__(self, data_file):
        self.data = self.load_data(data_file)
        self.dataset = self.all_to_bio(self.data)
    
    def load_data(self, data_file):
        Data = {}
        with open (data_file, "rt", encoding="utf-8") as f:
            # 文本使用空行进行分割句子
            for idx, line in enumerate(f.read().split("\n\n")):
                if not line:
                    break
                sentence, labels = "", []
                for i, item in enumerate(line.split("\n")):
                    char, tag = item.split(" ")
                    sentence += char
                    if tag.startswith("B"):
                        labels.append([i, i, char, tag[2:]])   # Remove the B- or I-
                        categories.add(tag[2:])
                    elif tag.startswith("I"):
                        labels[-1][1] = i
                        labels[-1][2] += char
                Data[idx] = {
                    "sentence" : sentence,
                    "labels" : labels
                }
        return Data
    
    def span_to_bio(self, sentence, spans):
        tokens = list(sentence)
        labels = ["O"] * len(tokens)
        for start, end, text, token_type in spans:
            labels[start] = f"B-{token_type}"
            for i in range(start + 1, end + 1):
                labels[i] = f"I-{token_type}"
        return {"tokens": tokens, "labels": labels}
    
    def all_to_bio(self, data):
        dataset_list = []
        for idx, item in data.items():
            sentence = item["sentence"]
            spans = item["labels"]
            bio_item = self.span_to_bio(sentence, spans)
            dataset_list.append(bio_item)
        return dataset_list
    
    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        return self.dataset[idx]

In [39]:
train_path = "./state3/dataset/ner_data/medical.train"
test_path = "./state3/dataset/ner_data/medical.test"

train_data = MyDataProcFunc(train_path)
for i in  range(3):
 print(train_data[i])

print("="*500)
test_data = MyDataProcFunc(test_path)
for i in  range(3):
 print(test_data[i])

{'tokens': ['现', '头', '昏', '口', '苦'], 'labels': ['O', 'O', 'O', 'B-临床表现', 'I-临床表现']}
{'tokens': ['目', '的', '观', '察', '复', '方', '丁', '香', '开', '胃', '贴', '外', '敷', '神', '阙', '穴', '治', '疗', '慢', '性', '心', '功', '能', '不', '全', '伴', '功', '能', '性', '消', '化', '不', '良', '的', '临', '床', '疗', '效'], 'labels': ['O', 'O', 'O', 'O', 'B-中医治疗', 'I-中医治疗', 'I-中医治疗', 'I-中医治疗', 'I-中医治疗', 'I-中医治疗', 'I-中医治疗', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-西医诊断', 'I-西医诊断', 'I-西医诊断', 'I-西医诊断', 'I-西医诊断', 'I-西医诊断', 'I-西医诊断', 'I-西医诊断', 'I-西医诊断', 'I-西医诊断', 'I-西医诊断', 'I-西医诊断', 'I-西医诊断', 'O', 'O', 'O', 'O', 'O']}
{'tokens': ['舒', '肝', '和', '胃', '消', '痞', '汤', '；', '功', '能', '性', '消', '化', '不', '良'], 'labels': ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-西医诊断', 'I-西医诊断', 'I-西医诊断', 'I-西医诊断', 'I-西医诊断', 'I-西医诊断', 'I-西医诊断']}
{'tokens': ['药', '进', '１', '０', '帖', '，', '黄', '疸', '稍', '退', '，', '饮', '食', '稍', '增', '，', '精', '神', '稍', '振'], 'labels': ['O', 'O', 'O', 'O', 'O', 'O', 'B-中医诊断', 'I-中医诊断', 'O', 'O', 'O', 'O', 'O', '

In [41]:
len(train_data), len(test_data)

(5259, 658)